In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (SATPdb)

This notebook processes and standardizes the **SATPdb** dataset by parsing toxic peptide sequences, resolving duplicate entries, generating dataset-level metadata, and exporting a clean dataset suitable for downstream machine learning and statistical analyses.

 - **Toxic effect / endpoint:** toxic
- **Source:** SATPdb
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Parses SATPdb toxic peptide sequences** from FASTA files, handling non-standard characters when present.
- **Assigns toxicity labels**:
  - all sequences in this source are labeled as toxic (`label = 1`).
- **Performs duplicate sequence checks**:
  - consistent duplicates are merged,
  - conflicting or ambiguous entries are flagged as erroneous sequences.
- **Generates dataset metadata** using the centralized raw data description file.
- **Exports curated outputs**:
  - `processed_toxic_dataset.csv`
  - `metadata.json`

In [2]:
name_source = "SATPdb"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
df_satpdb = (
        read_fasta_with_strange_character(f"{PATH_INPUT}/{name_source}/toxic.fasta")
        .assign(label = 1)
        [["sequence", "label"]]
    )

- Checking duplicates

In [4]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df_satpdb, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [5]:
df_full.shape

(4130, 2)

In [6]:
df_errors.shape

(0, 1)

- Working with metada

In [7]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [8]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df_satpdb)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Database',
 'static-dynamic': 'Dynamic',
 'license': 'No information',
 'year of publication': 2015,
 'last update date': datetime.datetime(2015, 11, 1, 0, 0),
 'download date': Timestamp('2025-08-25 00:00:00'),
 'file format': 'fasta',
 'peptide property': 'toxic',
 'dataset information': 'Positive',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'No information',
 'repository or server': 'https://webs.iiitd.edu.in/raghava/satpdb/down.php',
 'publication': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC4702810/',
 'number_of_raw_sequences': 4273,
 'number_of_sequences_retained': 4130,
 'number_of_positive_sequences': 4130,
 'number_of_negative_sequences': 0,
 'number_of_erroneous_sequences': 0,
 'modified_sequences_included': False}

- Exporting data

In [9]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [10]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_toxic_dataset.csv", index=False)